# Data Quality Audit - Customer Transactions
This notebook performs a comprehensive data quality audit on customer transaction data.

In [2]:
import pandas as pd
import numpy as np

# Load the dataset
data_path = "Dataset\week2_customer_transactions_messy.csv"
df = pd.read_csv(data_path)
print('Loaded:', data_path)
print('Shape:', df.shape)
df.head()

Loaded: Dataset\week2_customer_transactions_messy.csv
Shape: (11, 9)


,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
0,T0001,C100,2026-01-05,120.50,EUR,card,completed,DE,2026-01-05
1,T0002,C101,2026/01/06,0.00,EUR,CARD,completed,de,2026-01-20
2,T0003,C102,06-01-2026,-35.00,USD,bank_transfer,completed,US,2026-01-07
3,T0004,NaN,2026-01-07,250.00,EUR,card,pending,FR,2026-01-08
4,T0005,C104,2026-01-07,89.99,EURO,cash,completed,DE,2026-01-09


## Issue Detection and Classification

In [3]:
# FIXED: Dynamic issue_table based on actual data analysis
issues = []

# Check for missing values
for col in df.columns:
    missing_count = df[col].isna().sum()
    if missing_count > 0:
        issues.append({
            'Issue': f'Missing {col}',
            'Dimension': 'Completeness',
            'Impact': f'{missing_count} rows affected - impacts data reliability and analytics'
        })

# Check for duplicates
duplicate_count = df.duplicated(subset=['transaction_id']).sum()
if duplicate_count > 0:
    issues.append({
        'Issue': 'Duplicate transaction_id',
        'Dimension': 'Uniqueness',
        'Impact': f'{duplicate_count} duplicates found - may double count revenue'
    })

# Check for negative amounts
negative_amounts = (pd.to_numeric(df['amount'], errors='coerce') < 0).sum()
if negative_amounts > 0:
    issues.append({
        'Issue': 'Negative transaction amounts',
        'Dimension': 'Validity',
        'Impact': f'{negative_amounts} rows with negative values - violates business rules'
    })

# Check for zero amounts
zero_amounts = (pd.to_numeric(df['amount'], errors='coerce') == 0).sum()
if zero_amounts > 0:
    issues.append({
        'Issue': 'Zero transaction amounts',
        'Dimension': 'Validity',
        'Impact': f'{zero_amounts} rows with zero value - may be test data or errors'
    })

# Check for invalid dates
invalid_dates = pd.to_datetime(df['transaction_date'], errors='coerce').isna().sum()
if invalid_dates > 0:
    issues.append({
        'Issue': 'Invalid transaction dates',
        'Dimension': 'Validity',
        'Impact': f'{invalid_dates} rows with unparseable dates - impacts time-series analysis'
    })

# Check for inconsistent formatting
if 'currency' in df.columns:
    currency_variations = df['currency'].value_counts()
    if len(currency_variations) > len(df['currency'].str.upper().unique()):
        issues.append({
            'Issue': 'Inconsistent currency formatting',
            'Dimension': 'Consistency',
            'Impact': 'Mixed case/spelling - complicates aggregation and reporting'
        })

if 'region' in df.columns:
    region_variations = df['region'].value_counts()
    if len(region_variations) > len(df['region'].str.upper().unique()):
        issues.append({
            'Issue': 'Inconsistent region formatting',
            'Dimension': 'Consistency',
            'Impact': 'Mixed case - complicates regional analytics'
        })

if 'payment_method' in df.columns:
    payment_variations = df['payment_method'].value_counts()
    if len(payment_variations) > len(df['payment_method'].str.lower().unique()):
        issues.append({
            'Issue': 'Inconsistent payment_method formatting',
            'Dimension': 'Consistency',
            'Impact': 'Mixed case - complicates payment analytics'
        })

# Create issue table
issue_table = pd.DataFrame(issues)
print(f"\nTotal Issues Detected: {len(issues)}")
issue_table


Total Issues Detected: 9


,Issue,Dimension,Impact
0,Missing customer_id,Completeness,1 rows affected - impacts data reliability and...
1,Missing amount,Completeness,1 rows affected - impacts data reliability and...
2,Missing payment_method,Completeness,1 rows affected - impacts data reliability and...
3,Missing last_updated,Completeness,1 rows affected - impacts data reliability and...
4,Duplicate transaction_id,Uniqueness,1 duplicates found - may double count revenue
5,Negative transaction amounts,Validity,1 rows with negative values - violates busines...
6,Zero transaction amounts,Validity,1 rows with zero value - may be test data or e...
7,Invalid transaction dates,Validity,3 rows with unparseable dates - impacts time-s...
8,Inconsistent region formatting,Consistency,Mixed case - complicates regional analytics


## Key Performance Indicators (KPIs)

In [4]:
# Calculate KPIs
kpis = {}
kpis['Completeness Rate'] = 1 - (df.isna().sum().sum() / (df.shape[0] * df.shape[1]))
kpis['Duplication Rate'] = df.duplicated(subset=['transaction_id']).mean()

amount = pd.to_numeric(df['amount'], errors='coerce')
date_ok = pd.to_datetime(df['transaction_date'], errors='coerce', format='mixed').notna()
kpis['Error Rate'] = (amount.isna() | (amount < 0) | ~date_ok).mean()

pd.DataFrame({'KPI': list(kpis.keys()), 'Value': list(kpis.values())})

,KPI,Value
0,Completeness Rate,0.959596
1,Duplication Rate,0.090909
2,Error Rate,0.272727


## Comprehensive Audit Summary

In [5]:

audit_issues = []

# Missing customer_id
missing_customer = df['customer_id'].isna().sum()
if missing_customer > 0:
    audit_issues.append({
        'issue_type': 'Missing customer_id',
        'affected_rows': missing_customer,
        'severity': 'High',
        'recommended_next_action': 'Investigate source system; consider marking as anonymous or guest transactions'
    })

# Duplicate transaction_id
duplicates = df[df.duplicated(subset=['transaction_id'], keep=False)]
if len(duplicates) > 0:
    audit_issues.append({
        'issue_type': 'Duplicate transaction_id',
        'affected_rows': len(duplicates),
        'severity': 'Critical',
        'recommended_next_action': 'Review duplicate records; implement deduplication logic; investigate data ingestion pipeline'
    })

# Negative amounts
negative_amt = (pd.to_numeric(df['amount'], errors='coerce') < 0).sum()
if negative_amt > 0:
    audit_issues.append({
        'issue_type': 'Negative transaction amounts',
        'affected_rows': negative_amt,
        'severity': 'High',
        'recommended_next_action': 'Verify if refunds/chargebacks should be separate records; apply abs() or flag for review'
    })

# Zero amounts
zero_amt = (pd.to_numeric(df['amount'], errors='coerce') == 0).sum()
if zero_amt > 0:
    audit_issues.append({
        'issue_type': 'Zero transaction amounts',
        'affected_rows': zero_amt,
        'severity': 'Medium',
        'recommended_next_action': 'Clarify business rules for zero-value transactions; filter or flag test data'
    })

# Invalid dates
invalid_dt = pd.to_datetime(df['transaction_date'], errors='coerce').isna().sum()
if invalid_dt > 0:
    audit_issues.append({
        'issue_type': 'Invalid transaction dates',
        'affected_rows': invalid_dt,
        'severity': 'Critical',
        'recommended_next_action': 'Standardize date format in source; implement parsing rules; reject invalid dates at ingestion'
    })

# Inconsistent date formats (even if parseable)
date_formats = df['transaction_date'].astype(str).str.replace(r'\d', 'X', regex=True).value_counts()
if len(date_formats) > 1:
    audit_issues.append({
        'issue_type': 'Inconsistent date formats',
        'affected_rows': len(df),
        'severity': 'Medium',
        'recommended_next_action': 'Standardize to single format (e.g., YYYY-MM-DD); update data validation rules'
    })

# Missing amounts
missing_amt = df['amount'].isna().sum()
if missing_amt > 0:
    audit_issues.append({
        'issue_type': 'Missing transaction amount',
        'affected_rows': missing_amt,
        'severity': 'Critical',
        'recommended_next_action': 'Reject transactions without amounts; investigate source system data capture'
    })

# Missing payment_method
if 'payment_method' in df.columns:
    missing_pm = df['payment_method'].isna().sum()
    if missing_pm > 0:
        audit_issues.append({
            'issue_type': 'Missing payment_method',
            'affected_rows': missing_pm,
            'severity': 'Medium',
            'recommended_next_action': 'Set default value or mark as "unknown"; enhance data collection process'
        })

# Missing last_updated
if 'last_updated' in df.columns:
    missing_lu = df['last_updated'].isna().sum()
    if missing_lu > 0:
        audit_issues.append({
            'issue_type': 'Missing last_updated timestamp',
            'affected_rows': missing_lu,
            'severity': 'Low',
            'recommended_next_action': 'Auto-populate with current timestamp; implement audit trail'
        })

# Invalid status values
if 'status' in df.columns:
    valid_statuses = ['completed', 'pending', 'failed']  # case-insensitive check
    invalid_status = ~df['status'].str.lower().isin(valid_statuses)
    invalid_status_count = invalid_status.sum()
    if invalid_status_count > 0:
        audit_issues.append({
            'issue_type': 'Invalid status values',
            'affected_rows': invalid_status_count,
            'severity': 'High',
            'recommended_next_action': 'Define allowed status values; map or reject invalid statuses; update documentation'
        })

# Inconsistent currency codes
if 'currency' in df.columns:
    currency_upper = df['currency'].str.upper().value_counts()
    currency_actual = df['currency'].value_counts()
    if len(currency_actual) > len(currency_upper):
        affected = len(df) - df[df['currency'] == df['currency'].str.upper()].shape[0]
        audit_issues.append({
            'issue_type': 'Inconsistent currency formatting',
            'affected_rows': affected,
            'severity': 'Medium',
            'recommended_next_action': 'Normalize to uppercase ISO codes; validate against ISO 4217 standard'
        })

# Inconsistent region codes
if 'region' in df.columns:
    region_upper = df['region'].str.upper().value_counts()
    region_actual = df['region'].value_counts()
    if len(region_actual) > len(region_upper):
        affected = len(df) - df[df['region'] == df['region'].str.upper()].shape[0]
        audit_issues.append({
            'issue_type': 'Inconsistent region formatting',
            'affected_rows': affected,
            'severity': 'Low',
            'recommended_next_action': 'Standardize to uppercase ISO country codes; implement validation'
        })

# Inconsistent payment_method
if 'payment_method' in df.columns:
    pm_lower = df['payment_method'].str.lower().value_counts()
    pm_actual = df['payment_method'].value_counts()
    if len(pm_actual) > len(pm_lower):
        affected = len(df) - df[df['payment_method'] == df['payment_method'].str.lower()].shape[0]
        audit_issues.append({
            'issue_type': 'Inconsistent payment_method formatting',
            'affected_rows': affected,
            'severity': 'Low',
            'recommended_next_action': 'Standardize to lowercase; create enumeration of valid payment methods'
        })

# Outliers in amount (very large values)
amount_numeric = pd.to_numeric(df['amount'], errors='coerce')
if amount_numeric.notna().any():
    q1 = amount_numeric.quantile(0.25)
    q3 = amount_numeric.quantile(0.75)
    iqr = q3 - q1
    outliers = ((amount_numeric < (q1 - 3 * iqr)) | (amount_numeric > (q3 + 3 * iqr))).sum()
    if outliers > 0:
        audit_issues.append({
            'issue_type': 'Statistical outliers in amount',
            'affected_rows': outliers,
            'severity': 'Medium',
            'recommended_next_action': 'Review for potential fraud or data entry errors; implement business rule limits'
        })

# Create audit summary
audit_summary = pd.DataFrame(audit_issues)

if len(audit_summary) == 0:
    audit_summary = pd.DataFrame([{
        'issue_type': 'No issues detected',
        'affected_rows': 0,
        'severity': 'None',
        'recommended_next_action': 'Continue monitoring data quality'
    }])

print(f"\nTotal Audit Issues: {len(audit_issues)}")
print(f"Critical: {len(audit_summary[audit_summary['severity'] == 'Critical'])}")
print(f"High: {len(audit_summary[audit_summary['severity'] == 'High'])}")
print(f"Medium: {len(audit_summary[audit_summary['severity'] == 'Medium'])}")
print(f"Low: {len(audit_summary[audit_summary['severity'] == 'Low'])}")
audit_summary


Total Audit Issues: 13
Critical: 3
High: 3
Medium: 4
Low: 3


,issue_type,affected_rows,severity,recommended_next_action
0,Missing customer_id,1,High,Investigate source system; consider marking as...
1,Duplicate transaction_id,2,Critical,Review duplicate records; implement deduplicat...
2,Negative transaction amounts,1,High,Verify if refunds/chargebacks should be separa...
3,Zero transaction amounts,1,Medium,Clarify business rules for zero-value transact...
4,Invalid transaction dates,3,Critical,Standardize date format in source; implement p...
5,Inconsistent date formats,11,Medium,"Standardize to single format (e.g., YYYY-MM-DD..."
6,Missing transaction amount,1,Critical,Reject transactions without amounts; investiga...
7,Missing payment_method,1,Medium,"Set default value or mark as ""unknown""; enhanc..."
8,Missing last_updated timestamp,1,Low,Auto-populate with current timestamp; implemen...
9,Invalid status values,2,High,Define allowed status values; map or reject in...


## Rule-Based Validation

In [6]:

rules = {
    'transaction_id_required': df['transaction_id'].isna() | (df['transaction_id'].astype(str).str.strip() == ''),
    'customer_id_required': df['customer_id'].isna() | (df['customer_id'].astype(str).str.strip() == ''),
    'amount_non_negative': pd.to_numeric(df['amount'], errors='coerce') < 0,
    'amount_non_zero': pd.to_numeric(df['amount'], errors='coerce') == 0,
    'transaction_date_valid': pd.to_datetime(df['transaction_date'], errors='coerce').isna(),
    'status_valid': ~df['status'].str.lower().isin(['completed', 'pending', 'failed']),
    'currency_required': df['currency'].isna() | (df['currency'].astype(str).str.strip() == ''),
    'payment_method_required': df['payment_method'].isna() | (df['payment_method'].astype(str).str.strip() == ''),
}

def summarize_rule_violations(rule_dictionary):
    return pd.DataFrame({
        'rule_name': list(rule_dictionary.keys()),
        'affected_rows': [int(mask.sum()) for mask in rule_dictionary.values()]
    })

rule_summary = summarize_rule_violations(rules)
print(f"\nRules with violations: {(rule_summary['affected_rows'] > 0).sum()}")
rule_summary


Rules with violations: 6


,rule_name,affected_rows
0,transaction_id_required,0
1,customer_id_required,1
2,amount_non_negative,1
3,amount_non_zero,1
4,transaction_date_valid,3
5,status_valid,2
6,currency_required,0
7,payment_method_required,1


## Detailed Data Profiling

In [7]:
# Show records with issues
# print("\n=== Records with Data Quality Issues ===")

# Show duplicates
if len(duplicates) > 0:
    print(f"\nDuplicate Transaction IDs:")
    print(duplicates[['transaction_id', 'customer_id', 'transaction_date', 'amount']])

# Show missing customer IDs
missing_cust = df[df['customer_id'].isna()]
if len(missing_cust) > 0:
    print(f"\nMissing Customer IDs:")
    print(missing_cust[['transaction_id', 'transaction_date', 'amount', 'status']])

# Show negative amounts
neg_amounts = df[pd.to_numeric(df['amount'], errors='coerce') < 0]
if len(neg_amounts) > 0:
    print(f"\nNegative Amounts:")
    print(neg_amounts[['transaction_id', 'customer_id', 'amount', 'status']])

# Show invalid dates
invalid_dates_df = df[pd.to_datetime(df['transaction_date'], errors='coerce').isna()]
if len(invalid_dates_df) > 0:
    print(f"\nInvalid Dates:")
    print(invalid_dates_df[['transaction_id', 'customer_id', 'transaction_date']])


Duplicate Transaction IDs:
  transaction_id customer_id transaction_date  amount
5          T0006        C105       2026-01-08   19.99
6          T0006        C105       2026-01-08   19.99

Missing Customer IDs:
  transaction_id transaction_date  amount   status
3          T0004       2026-01-07   250.0  pending

Negative Amounts:
  transaction_id customer_id  amount     status
2          T0003        C102   -35.0  completed

Invalid Dates:
  transaction_id customer_id transaction_date
1          T0002        C101       2026/01/06
2          T0003        C102       06-01-2026
7          T0007        C106       2026-02-30


## Summary Statistics

In [8]:
# print("\n=== Data Quality Summary ===")
print(f"Total Records: {len(df)}")
print(f"Total Columns: {len(df.columns)}")
print(f"\nData Quality Metrics:")
print(f"  Completeness Rate: {kpis['Completeness Rate']:.2%}")
print(f"  Duplication Rate: {kpis['Duplication Rate']:.2%}")
print(f"  Error Rate: {kpis['Error Rate']:.2%}")
print(f"\nIssues Summary:")
print(f"  Total Issues Identified: {len(audit_summary)}")
print(f"  Total Affected Rows: {audit_summary['affected_rows'].sum()}")
print(f"\nRecommendation: Address Critical and High severity issues first.")

Total Records: 11
Total Columns: 9

Data Quality Metrics:
  Completeness Rate: 95.96%
  Duplication Rate: 9.09%
  Error Rate: 27.27%

Issues Summary:
  Total Issues Identified: 13
  Total Affected Rows: 28

Recommendation: Address Critical and High severity issues first.


KPI SUMMARY

Completeness rate is 95.96%.
duplication rate is 9.09%
error rate is 27.27%
Dataset is weak in quality nearly 1/3 rows have errors.

AUDIT FINDINGS

Found out there is identical rows, invalid dates, negative amounts.
consistent problems like different currency, normalization, invalid status.

RECOMMENDED CLEANING ACTIONS

Drop duplicates, fix date formats, handle negative amounts, text normalization